# 24 — PyCaret Model Screening for Classification (Objective 2)

**Objective.** Use PyCaret to screen standard classification models for the **binary classification task** (Not High Risk vs High Risk) on the existing training rows, then select one model family for tuning in the next step.

**Input.** `data/processed/classification_model_input.csv`; `feature_engine/model_features.csv`.

**Output.** `training_and_evaluation/pycaret_model_screening.csv`; `training_and_evaluation/pycaret_selected_model.csv`.

This notebook screens model families for binary classification and checks whether resampling is needed for the selected classifier.

## 0. Setup

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *
set_style()

In [ ]:
import pandas as pd
import pycaret
from pycaret.classification import ClassificationExperiment
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import AdaBoostClassifier
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

print("PyCaret", pycaret.__version__)

## 1. Load the fixed training split

This section checks that PyCaret only receives the 20,000 training rows from the fixed split. The 5,000 test rows are not passed to PyCaret.

In [ ]:
p = obj_paths(2)
target = "breakdown_risk"

data = pd.read_csv(p["processed"] / "classification_model_input.csv")
features = pd.read_csv(p["feature_engine"] / "model_features.csv")["feature"].tolist()

train = data.loc[data["split"].eq("train")].copy()
test = data.loc[data["split"].eq("test")].copy()

print("training rows:", len(train))
print("test rows kept untouched:", len(test))
print("feature count:", len(features))
print("class counts in training:")
print(train[target].value_counts().reindex(["Not High Risk", "High Risk"]))

> **Interpretation.**
>
> - PyCaret screens models on the existing training set only.
> - The fixed test set remains outside the screening process.
> - The binary target is approximately balanced (Not High Risk ~52%, High Risk ~48%), so macro-F1 captures both classes equally without requiring separate imbalance treatment at this stage.

## 2. PyCaret model screening

**Rule fixed before the run.**

- Compare only the standard model families named in the project scope.
- Add **Macro F1** to give equal weight to both the Not High Risk and High Risk classes in the binary task.
- Rank by Macro F1.
- Select the top-ranked model family for tuning in the next step.
- Use PyCaret normalization during screening so LR/SVM are not disadvantaged by scale.

In [ ]:
screen_train = train[features + [target]].copy()

exp = ClassificationExperiment(
    target=target,
    session_id=RANDOM_STATE,
    train_size=0.8,
    fold=5,
    preprocess=True,
    normalize=True,
    verbose=False,
    n_jobs=1,
).fit(screen_train)

exp.add_metric(
    "macro_f1",
    "Macro F1",
    lambda y_true, y_pred: f1_score(y_true, y_pred, average="macro"),
    greater_is_better=True,
)

include_models = ["lr", "nb", "dt", "rf", "svm", "ada", "xgboost", "catboost"]
result = exp.compare_models(
    include=include_models,
    sort="Macro F1",
    n_select=len(include_models),
    turbo=False,
    errors="raise",
    verbose=False,
)

leaderboard = result.leaderboard.copy()
model_names = exp.models()[["Name"]].rename(columns={"Name": "model_name"})
leaderboard = leaderboard.merge(model_names, left_on="Model", right_index=True, how="left")
ordered_cols = ["Model", "model_name"] + [c for c in leaderboard.columns if c not in ["Model", "model_name"]]
leaderboard = leaderboard[ordered_cols]
print(leaderboard.to_string(index=False))

selected = leaderboard.iloc[0]
selected_model = pd.DataFrame([
    {
        "selected_model_id": selected["Model"],
        "selected_model_name": selected["model_name"],
        "selection_metric": "Macro F1",
        "selection_metric_value": selected["Macro F1"],
        "pycaret_train_size_inside_training_rows": 0.8,
        "folds": 5,
        "screening_scope": "existing Objective 2 training rows only",
    }
])

save_table(leaderboard, p["train_eval"] / "pycaret_model_screening.csv", index=False)
save_table(selected_model, p["train_eval"] / "pycaret_selected_model.csv", index=False)

> **Interpretation.**
>
> - The leaderboard (above) ranks all eight candidate models by macro-F1 on the binary classification task (Not High Risk vs High Risk).
> - Ada Boost, CatBoost, Random Forest, and Logistic Regression form a cluster at the top, all within about 0.005 macro-F1 of each other.
> - Gaussian Naive Bayes ranks last with a much lower macro-F1 (around 0.40), which reflects its tendency to collapse predictions towards one class on this binary feature set.
> - The decision tree sits second to last — it is clearly weaker than the ensemble methods and Logistic Regression.

> **Decision — model selection.**
>
> - PyCaret ranks Ada Boost Classifier first by Macro F1 for binary classification (as shown in the leaderboard above).
> - Ada Boost is selected for binary classification: it ranks first on macro-F1, is a well-understood ensemble method, and is interpretable through its per-estimator structure.
> - The tuning step uses Optuna to tune the selected Ada Boost model family.

## 3. Selected-model imbalance check

After PyCaret selects Ada Boost, the imbalance treatment is checked on that same model family.

**With a near-balanced binary target (~52% Not High Risk / ~48% High Risk), no resampling is expected. This section confirms by testing random over- and undersampling.**

**Rule fixed before the run.**

- Compare no resampling, random oversampling and random undersampling.
- Use 5-fold training macro-F1 (binary).
- Keep resampling only if it improves macro-F1.
- Class weighting is not tested here because AdaBoost does not expose a class-weight parameter directly.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_train = train[features]
y_train = train[target].astype(str)

imbalance_models = {
    "no_resampling": AdaBoostClassifier(random_state=RANDOM_STATE),
    "random_oversampling": ImbPipeline([
        ("sampler", RandomOverSampler(random_state=RANDOM_STATE)),
        ("model", AdaBoostClassifier(random_state=RANDOM_STATE)),
    ]),
    "random_undersampling": ImbPipeline([
        ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),
        ("model", AdaBoostClassifier(random_state=RANDOM_STATE)),
    ]),
}

imbalance_rows = []
for treatment, model in imbalance_models.items():
    scores = cross_val_score(model, X_train, y_train, scoring="f1_macro", cv=cv, n_jobs=1)
    imbalance_rows.append(
        {
            "model": "AdaBoost",
            "treatment": treatment,
            "macro_f1_mean": scores.mean(),
            "macro_f1_std": scores.std(),
        }
    )

imbalance_probe = pd.DataFrame(imbalance_rows).sort_values("macro_f1_mean", ascending=False)
print(imbalance_probe.round(4).to_string(index=False))
save_table(imbalance_probe, p["train_eval"] / "imbalance_probe.csv", index=False)

> **Interpretation.**
>
> - All three resampling treatments produce macro-F1 values within a narrow range (as computed above).
> - No resampling achieves the highest training macro-F1, consistent with the near-balanced binary target.
> - The small standard deviations confirm that results are stable across folds.
> - The approximately balanced binary target (~52/48) does not require special resampling treatment for Ada Boost on this data.

> **Decision — class imbalance handling.**
>
> - Ada Boost with no resampling achieves the best binary macro-F1 (as computed above).
> - Both resampling strategies reduce macro-F1.
> - **No resampling** — the binary target is approximately balanced (~52/48), and no resampling strategy improves performance.
> - The imbalance is managed through stratified splitting, macro-F1 as the selection metric, and per-class reporting in the evaluation step.

## 4. Checks

In [ ]:
screen_path = p["train_eval"] / "pycaret_model_screening.csv"
selected_path = p["train_eval"] / "pycaret_selected_model.csv"
imbalance_path = p["train_eval"] / "imbalance_probe.csv"

check_screen = pd.read_csv(screen_path)
check_selected = pd.read_csv(selected_path)
imbalance_probe = pd.read_csv(imbalance_path)

assert len(check_screen) == 8
assert check_selected.loc[0, "selected_model_id"] == "ada"
assert set(check_screen["Model"]) == set(include_models)
assert imbalance_probe.shape == (3, 4), f"expected (3, 4), got {imbalance_probe.shape}"
assert imbalance_probe.isna().sum().sum() == 0, "null values found in imbalance_probe"
print("checks passed")

---
## Summary

PyCaret screened eight model families on the Objective 2 training rows for the **binary classification task** (Not High Risk vs High Risk) and ranked Ada Boost Classifier first by macro-F1. CatBoost, Random Forest, and Logistic Regression followed within 0.005 macro-F1 of each other, while Gaussian Naive Bayes ranked last — it collapsed its predictions towards one class on the binary feature set, producing a much lower macro-F1. Ada Boost was selected for tuning because it ranked first on the primary metric.

An imbalance probe then compared three treatments on the selected model. No resampling produced the best binary macro-F1, so no resampling strategy is applied. The binary target is approximately balanced (~52% Not High Risk / ~48% High Risk), confirming that no special treatment is needed. The class balance is managed through stratified splitting, macro-F1 as the selection metric, and per-class reporting in the evaluation step.